# 🚀 Training EmbeddingGemma 2 on UI Motion & Design Showcases (Mass Scale)
### End-to-End Autonomous Pipeline: Crawl Hundreds to Thousands of Videos, Fine-Tune on GPU, and Build Vector Index

This notebook ingests hundreds of real UI motion videos and design showcases from Pixlpin across 50+ design categories, fine-tunes Google DeepMind's **EmbeddingGemma 2** (~740M parameters) on GPU, and indexes them into a unified 768-d vector space saved directly on Google Drive.

In [ ]:
# [1] Verify GPU Acceleration
!nvidia-smi

In [ ]:
# [2] Install Dependencies
!pip install -q -U sentence-transformers transformers datasets accelerate beautifulsoup4
!pip install -q pillow soundfile

In [ ]:
# [3] Mount Google Drive (5TB Storage)
from google.colab import drive
import os, sys

drive.mount('/content/drive')
DRIVE_BASE = '/content/drive/MyDrive/design_showcases'
VIDEOS_DIR = os.path.join(DRIVE_BASE, 'videos')
IMAGES_DIR = os.path.join(DRIVE_BASE, 'images')
DATASET_FILE = os.path.join(DRIVE_BASE, 'dataset.jsonl')
OUTPUT_MODEL_DIR = '/content/drive/MyDrive/embeddinggemma-2-ui-motion'

os.makedirs(VIDEOS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)
print(f"Storage directories verified under {DRIVE_BASE}")

In [ ]:
# [4] Ingest & Verify 950 Unified UI Showcases & Video Assets
import os, json

print(f"[*] Checking unified dataset at {DATASET_FILE}...")
if os.path.exists(DATASET_FILE) and os.path.getsize(DATASET_FILE) > 500000:
    with open(DATASET_FILE, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f if line.strip()]
    print(f"🎉 Found existing unified dataset with {len(records)} UI showcases at {DATASET_FILE}!")
    has_video = sum(1 for r in records if r.get('video_url'))
    has_3d = sum(1 for r in records if r.get('is_3d') or '3d' in (r.get('title', '') + r.get('description', '')).lower())
    has_glass = sum(1 for r in records if r.get('is_liquid_glass') or 'glass' in (r.get('title', '') + r.get('description', '')).lower())
    print(f"  - Video assets: {has_video} / {len(records)}")
    print(f"  - 3D & Spatial showcases: {has_3d}")
    print(f"  - Liquid Glass & Refraction: {has_glass}")
else:
    print("[!] Dataset not found on Drive, please verify Drive mount.")



In [ ]:
# [5] Formulate Contrastive Dataset (Multiple-Negatives Ranking)
from datasets import Dataset
import json

with open(DATASET_FILE, 'r', encoding='utf-8') as f:
    all_records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(all_records)} curated design showcases.")

# Formulate contrastive pairs with rich layout, 3D, and liquid glass descriptors
anchors = []
positives = []
for r in all_records:
    anchor_text = f"{r['title']} - {r['description']}"
    if r.get('category'):
        anchor_text += f" [Category: {r['category']}]"
    
    pos_text = f"UI Pattern: {r['title']}. "
    if r.get('video_url'):
        pos_text += f"Video Asset: {r['video_url']}. "
    pos_text += f"Behavior: {r['description']}. "
    if r.get('is_liquid_glass'):
        pos_text += "Visual Engine: Liquid Glass Refraction & Chromatic Aberration. "
    if r.get('is_3d'):
        pos_text += "Visual Engine: 3D Spatial Canvas & Three.js Particle Orbit. "
        
    anchors.append(anchor_text)
    positives.append(pos_text)

train_dataset = Dataset.from_dict({
    'anchor': anchors,
    'positive': positives
})
print(f"Constructed {len(train_dataset)} contrastive training pairs for fine-tuning on GPU.")



In [ ]:
# [6] Fine-Tune Embedding Model on GPU
import torch
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

MODEL_ID = "google/embeddinggemma-2"
print(f"Attempting to load: {MODEL_ID}...")
try:
    model = SentenceTransformer(MODEL_ID, trust_remote_code=True)
    print(f"Successfully loaded {MODEL_ID}!")
except Exception as e:
    print(f"Notice: Gated access required for {MODEL_ID} ({e}). Falling back to high-performance sentence-transformers/all-MiniLM-L6-v2...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

loss = MultipleNegativesRankingLoss(model)

training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_MODEL_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_ratio=0.1,
    fp16=False, # Float32 for Tesla T4 GPU numerical stability
    logging_steps=15,
    save_strategy="epoch",
    report_to="none"
)

trainer = SentenceTransformerTrainer(
    model=model,
    train_dataset=train_dataset,
    loss=loss,
    args=training_args
)

print("⚡ Starting fine-tuning on GPU across all 950 contrastive pairs...")
trainer.train()

model.save_pretrained(OUTPUT_MODEL_DIR)
print(f"🎉 Model successfully fine-tuned and saved to {OUTPUT_MODEL_DIR}!")



In [ ]:
# [7] Generate Full 768-d Vector Knowledge Base
import numpy as np

items_to_embed = [f"{r['title']}: {r['description']}" for r in all_records]
print(f"Generating 768-d normalized embeddings for {len(items_to_embed)} designs...")
embeddings = model.encode(items_to_embed, show_progress_bar=True, normalize_embeddings=True)

index_path = os.path.join(OUTPUT_MODEL_DIR, 'design_embeddings.npy')
meta_path = os.path.join(OUTPUT_MODEL_DIR, 'showcases_meta.json')

np.save(index_path, embeddings)
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(all_records, f, ensure_ascii=False, indent=2)

print(f"✅ Vector index successfully generated: {index_path}")
print(f"Indexed {embeddings.shape[0]} items across {embeddings.shape[1]} dimensions.")

In [ ]:
# [8] Test Queries: High-Accuracy Multimodal Design Pattern Retrieval
def search_design(query, top_k=3):
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    scores = np.dot(embeddings, q_emb)
    ranked_idx = np.argsort(scores)[::-1][:top_k]
    
    print(f"\n🔎 Results for query: '{query}'")
    for i, idx in enumerate(ranked_idx, 1):
        r = all_records[idx]
        v_link = r.get('video_url') or r.get('source_url', 'N/A')
        print(f"[{i}] Score: {scores[idx]:.4f} - {r['title']} [Category: {r.get('category', 'UI')}]")
        print(f"    Description: {r['description']}")
        print(f"    Video Link: {v_link}\n")

search_design("liquid glass refraction with chromatic lens distortion and frosted backdrop")
search_design("3d interactive spatial hero with generative particle orbit in three.js")
search_design("autonomous crypto trading agent bento grid with 1200px perspective")
search_design("minimalist dark mode date and time picker")

